In [1]:
import pandas as pd
import json
import cv2
import numpy as np
from pathlib import Path
import os
from tqdm.notebook import tqdm

In [17]:
video_path = r"/home/share/schaer2/idtracking_keypoint/demo/video_25-09-05-14-40-56_c0.mp4"
bbox = r"/home/share/schaer2/idtracking_keypoint/demo/video_25-09-05-14-40-56_c0_bbox_0-206_20250905_151446.json"
skpoint = r"/home/share/schaer2/idtracking_keypoint/demo/results_skeleton_video_25-09-05-14-40-56_c0.json"
# timeline_behavior_path = r"/home/share/schaer2/idtracking_keypoint/output/8124_data_Timelight_segment.csv"
timeline_behavior_path = None

max_seconds = -1  # Set to -1 for full video duration


In [18]:
do_timeline = True if timeline_behavior_path else False

In [19]:
if do_timeline:
    # Load the CSV file using the correct path
    timeline_behavior_data = pd.read_csv(timeline_behavior_path)
    print("Timeline behavior data:")
    print(timeline_behavior_data.head())
    print(f"\nColumns: {list(timeline_behavior_data.columns)}")
    print(f"Shape: {timeline_behavior_data.shape}")
    timeline_behavior_data

else:
    timeline_behavior_data = None

In [20]:
# if do_timeline:
#     # Show basic structure without the full data
#     print("Basic data info:")
#     print(f"Shape: {timeline_behavior_data.shape}")
#     print(f"Columns: {list(timeline_behavior_data.columns)}")
#     print("\nFirst few rows (limited columns):")
#     if len(timeline_behavior_data.columns) > 5:
#         print(timeline_behavior_data[timeline_behavior_data.columns[:5]].head())
#     else:
#         print(timeline_behavior_data.head())

#     # Look for relevant columns for behaviors
#     behavior_cols = [col for col in timeline_behavior_data.columns if any(word in col.lower() for word in ['behavior', 'start', 'end', 'left', 'right', 'shoulder'])]
#     print(f"\nBehavior-related columns: {behavior_cols}")

#     if behavior_cols:
#         print("\nSample behavior data:")
#         print(timeline_behavior_data[behavior_cols].head())

In [21]:
# if do_timeline:
#     # Check the modifier column for body parts
#     print("Unique modifiers (body parts):")
#     print(timeline_behavior_data['modifier'].unique())

#     print("\nBehavior data grouped by modifier:")
#     for modifier in timeline_behavior_data['modifier'].unique():
#         data = timeline_behavior_data[timeline_behavior_data['modifier'] == modifier]
#         print(f"\n{modifier}: {len(data)} events")
#         print(data[['start', 'end']].head(3))

In [ ]:
# Enhanced implementation with timeline and behavior annotations

# OpenPose COCO connections for skeleton
COCO_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
]

# Colors for 3 individuals (BGR format)
COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255,255,0), (0,255,255), (255,0,255), (255,255,255), (0,0,0)]  # Blue, Green, Red

# Behavior colors
BEHAVIOR_COLORS = {
    'Bras Gauche': (0, 0, 255),    # Red for left arm
    'Bras Droit': (255, 0, 0),     # Blue for right arm
    'Shoulder': (0, 255, 0)         # Green for shoulder (if present)
}

def draw_skeleton(image, keypoints, connections, color, threshold=0.3):
    """Draw skeleton keypoints and connections"""
    overlay = image.copy()
    h, w = overlay.shape[:2]
    
    if keypoints is None or len(keypoints) == 0:
        return overlay
    
    # Draw connections
    for connection in connections:
        idx1, idx2 = connection
        if (idx1 < len(keypoints) and idx2 < len(keypoints) and 
            keypoints[idx1][2] > threshold and keypoints[idx2][2] > threshold):
            x1, y1 = int(keypoints[idx1][0]), int(keypoints[idx1][1])
            x2, y2 = int(keypoints[idx2][0]), int(keypoints[idx2][1])
            cv2.line(overlay, (x1, y1), (x2, y2), color, 4)
    
    # Draw keypoints
    for x, y, conf in keypoints:
        if conf > threshold:
            cv2.circle(overlay, (int(x), int(y)), 6, color, -1)

    # Blend the overlay with the original image
    alpha = 0.5  # Transparency factor (0.0 to 1.0)
    cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0, image)

    return image

def draw_bbox(image, bbox, color, is_normalized=True):
    """Draw bounding box"""
    overlay = image.copy()
    h, w = overlay.shape[:2]
    
    if bbox is None:
        return overlay

    x1 = bbox[0]
    y1 = bbox[1]
    x2 = bbox[2]
    y2 = bbox[3]
    
    if is_normalized:
        # Convert normalized coordinates to pixel values
        x1 = int(x1 * w)
        y1 = int(y1 * h)
        x2 = int(x2 * w)
        y2 = int(y2 * h)
        
        # Ensure coordinates are within image bounds
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w - 1))
        y2 = max(0, min(y2, h - 1))

    alpha = 0.5  # Transparency factor (0.0 to 1.0)

    cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

    # Blend the overlay with the original image
    cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0, image)
    return image

def draw_timeline_with_legends(width, height, current_time, total_duration, behavior_data, timeline_height=60):
    """Draw timeline with behavior annotations, time cursor, and legends"""
    # Create extended timeline with space for legends
    legend_height = 80
    total_height = timeline_height + legend_height
    timeline = np.zeros((total_height, width, 3), dtype=np.uint8)
    
    # Draw timeline background (dark gray)
    cv2.rectangle(timeline, (0, 0), (width, timeline_height), (40, 40, 40), -1)
    
    # Draw behavior rectangles
    for _, row in behavior_data.iterrows():
        start_time = row['start']
        end_time = row['end']
        modifier = row['modifier']
        
        # Convert time to pixel positions
        start_x = int((start_time / total_duration) * width)
        end_x = int((end_time / total_duration) * width)
        
        # Get color for behavior type
        color = BEHAVIOR_COLORS.get(modifier, (128, 128, 128))  # Default gray
        
        # Draw behavior rectangle
        cv2.rectangle(timeline, (start_x, 10), (end_x, timeline_height-10), color, -1)
        
        # Add behavior label
        label = modifier.replace('Bras ', '')  # Shorten label
        text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)[0]
        text_x = start_x + 2
        text_y = timeline_height // 2 + text_size[1] // 2
        if text_x + text_size[0] < end_x:  # Only draw if fits
            cv2.putText(timeline, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Draw time cursor (white vertical line)
    cursor_x = int((current_time / total_duration) * width)
    cv2.line(timeline, (cursor_x, 0), (cursor_x, timeline_height), (255, 255, 255), 2)
    
    # Draw time markers
    for i in range(0, int(total_duration), max(1, int(total_duration // 10))):
        marker_x = int((i / total_duration) * width)
        cv2.line(timeline, (marker_x, timeline_height-5), (marker_x, timeline_height), (200, 200, 200), 1)
        # Add time label
        time_label = f"{i//60}:{i%60:02d}" if i >= 60 else f"{i}s"
        cv2.putText(timeline, time_label, (marker_x+2, timeline_height-15), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (200, 200, 200), 1)
    
    # Draw legends below timeline
    legend_y_start = timeline_height + 10
    
    # Individual colors legend (left side)
    legend_x = 20
    cv2.putText(timeline, "Individuals:", (legend_x, legend_y_start), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    individual_labels = ["Child", "Clinician", "Parent"]
    for i, (color, label) in enumerate(zip(COLORS, individual_labels)):
        y_pos = legend_y_start + 20 + (i * 18)
        # Draw color rectangle
        cv2.rectangle(timeline, (legend_x, y_pos-8), (legend_x+15, y_pos+2), color, -1)
        # Draw label
        cv2.putText(timeline, label, (legend_x + 20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Behavior colors legend (right side)
    behavior_legend_x = width // 2 + 50
    cv2.putText(timeline, "Repetitive Behaviors:", (behavior_legend_x, legend_y_start), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    behavior_items = [
        ("Bras Gauche", "Left Arm", BEHAVIOR_COLORS.get('Bras Gauche', (128, 128, 128))),
        ("Bras Droit", "Right Arm", BEHAVIOR_COLORS.get('Bras Droit', (128, 128, 128))),
        ("Shoulder", "Shoulder", BEHAVIOR_COLORS.get('Shoulder', (128, 128, 128)))
    ]
    
    for i, (key, label, color) in enumerate(behavior_items):
        y_pos = legend_y_start + 20 + (i * 18)
        # Draw color rectangle
        cv2.rectangle(timeline, (behavior_legend_x, y_pos-8), (behavior_legend_x+15, y_pos+2), color, -1)
        # Draw label
        cv2.putText(timeline, label, (behavior_legend_x + 20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Add current time display (top right)
    time_text = f"Time: {int(current_time//60)}:{int(current_time%60):02d}"
    time_size = cv2.getTextSize(time_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    cv2.putText(timeline, time_text, (width - time_size[0] - 10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    return timeline

def extract_frame_data(bbox_data, keypoint_data, frame_idx):
    """Extract bbox and keypoints for all individuals in a frame"""
    # Extract bboxes
    bboxes = {}
    if str(frame_idx) in bbox_data:
        frame_bboxes = bbox_data[str(frame_idx)]
        bboxes = frame_bboxes
    
    # Extract keypoints
    keypoints_dict = {}
    if 'instance_info' in keypoint_data:
        # Find frame data
        frame_data = None
        for data in keypoint_data['instance_info']:
            if data['frame_id'] == frame_idx:
                frame_data = data
                break
        
        if frame_data and 'instances' in frame_data:
            instances = frame_data['instances']
            for i, person in enumerate(instances):
                if person is not None and 'keypoints' in person and 'keypoint_scores' in person and 'keypoints_label' in person:
                    kp = np.array(person['keypoints'])  # (17, 2)
                    scores = np.array(person['keypoint_scores'])  # (17,)
                    identity = str(int(person['keypoints_label']))  # (17,)
                    # Combine to (17, 3) format
                    combined = np.zeros((17, 3))
                    combined[:, :2] = kp
                    combined[:, 2] = scores
                    keypoints_dict[int(identity)] = combined
                else:
                    tqdm.write(f"Warning: Frame {frame_idx} individual {i} missing keypoints data.")
    
    return bboxes, keypoints_dict

def create_enhanced_video(video_path, bbox_data, keypoint_data, behavior_data, output_path, max_seconds=10, threshold=0.5):
    """Create enhanced video with timeline and behavior annotations"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    if max_seconds <= 0:
        print("Processing full video duration...")
        max_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total_duration = max_frames / fps
    else:
        max_frames = int(fps * max_seconds)
        total_duration = max_seconds
    
    print(f"Processing {max_frames} frames ({total_duration:.1f}s) at {fps} fps")
    
    # Timeline height (increased for legends)
    if do_timeline:
        timeline_height = 140  # 60 for timeline + 80 for legends
    else:
        timeline_height = 0
    
    # Output video dimensions: width*2 for 2x2 grid, height*2 + timeline_height
    # output_width = width * 2
    output_width = width 
    # output_height = height * 2 + timeline_height
    output_height = height
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (output_width, output_height))
    
    for frame_idx in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Current time in seconds
        current_time = frame_idx / fps
        
        # Get data for all individuals
        bboxes, keypoints_dict = extract_frame_data(bbox_data, keypoint_data, frame_idx)
        
        # Create 4 quadrants
        # top_left = frame.copy()  # Original
        # cv2.putText(top_left, f"Frame: {frame_idx}", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        # cv2.putText(top_left, f"Time: {current_time:.2f}s", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # top_right = frame.copy()  # With bboxes
        # for idx, (identity, bbox) in enumerate(bboxes.items()):
        #     if bbox is not None:
        #         # Use COLORS cycling if more individuals than colors
        #         color = COLORS[int(identity) % len(COLORS)] if len(COLORS) > 0 else (255,255,255)
        #         top_right = draw_bbox(top_right, bbox, color)

        #         cv2.putText(top_right, f"ID: {identity}", (10 + int(identity) * 50, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # bottom_left = np.zeros_like(frame)  # Keypoints on black
        # for idx, (identity, keypoints) in enumerate(keypoints_dict.items()):
        #     if keypoints is not None:
        #         color = COLORS[int(identity) % len(COLORS)] if len(COLORS) > 0 else (255,255,255)
        #         bottom_left = draw_skeleton(bottom_left, keypoints, COCO_CONNECTIONS, color)
        bottom_right = frame.copy()  # Keypoints on video
        for idx, (identity, keypoints) in enumerate(keypoints_dict.items()):
            if keypoints is not None:
                color = COLORS[int(identity) % len(COLORS)] if len(COLORS) > 0 else (255,255,255)
                bottom_right = draw_skeleton(bottom_right, keypoints, COCO_CONNECTIONS, color, threshold=threshold)
        
        # Combine into 2x2 grid
        # top_row = np.hstack((top_left, top_right))
        # bottom_row = np.hstack((bottom_left, bottom_right))
        # video_grid = np.vstack((top_row, bottom_row))
        # final_frame = bottom_left
        final_frame = bottom_right
        
        
        # Create timeline with legends
        # if do_timeline:
        #     timeline = draw_timeline_with_legends(output_width, timeline_height, current_time, total_duration, behavior_data)
        #     # Combine video grid with timeline (timeline in the middle)
        #     final_frame = np.vstack((video_grid[:height*2//2], timeline, video_grid[height*2//2:]))
        # else:
        #     # Combine video grid without timeline
        #     final_frame = np.vstack((video_grid[:height*2//2], video_grid[height*2//2:]))
        
        out.write(final_frame)
        
        if frame_idx % 1000 == 0:
            print(f"Processed {frame_idx}/{max_frames} frames")
    
    cap.release()
    out.release()
    print(f"✅ Enhanced video saved: {output_path}")

# Load data and create enhanced test video
print("Loading data...")
with open(bbox, 'r') as f:
    bbox_data = json.load(f)
with open(skpoint, 'r') as f:
    keypoint_data = json.load(f)


# Create output path
if max_seconds <= 0:
    output_path = str(Path(skpoint).parent / f"{os.path.basename(video_path).split('.')[0] }_skeleton_video_with_background_full.mp4")
else:
    output_path = str(Path(skpoint).parent / f"{os.path.basename(video_path).split('.')[0] }_skeleton_video_with_background_{max_seconds}s.mp4")

# max_seconds = 100
print("Creating 10-second enhanced video with timeline...")
create_enhanced_video(video_path, bbox_data, keypoint_data, timeline_behavior_data, output_path, max_seconds=max_seconds, threshold=0.55)

Loading data...
Creating 10-second enhanced video with timeline...
Processing full video duration...
Processing 206 frames (13.7s) at 14.999925364449036 fps
Processed 0/206 frames
✅ Enhanced video saved: /home/share/schaer2/idtracking_keypoint/demo/video_25-09-05-14-40-56_c0_skeleton_video_with_background_full.mp4
